## Vraag 1: Correlatie tussen de drie fondsen

**Vraag:** Bewegen IWDA, EMIM en IUSN (small cap) onafhankelijk genoeg om mijn 
small-cap- en EM-allocatie te rechtvaardigen, of bewegen ze sterk gecorreleerd 
mee met de wereldindex?

(IWDA volgt de wereldindex, het is niet een wereldindex zelf. Deze foute gedachtegang had ik zelf eerst ook :p vandaar deze vraagstelling.)

**Aanpak:** Van de slotkoersen (data/prices.csv) naar dagrendementen, en daarover 
een correlatiematrix berekenen.

**Op te zoeken pandas-operaties:**
- CSV inlezen met de datum als datetime-index
- Van koersen naar dagrendementen (procentuele verandering)
- Correlatie over meerdere kolommen in één keer

In [12]:
import pandas as pd

def load_data(csv_path: str) -> pd.DataFrame:
    """Function to load the dataframe by using the Date column as index and parsing dates to standardise."""
    return pd.read_csv(csv_path, index_col=0, parse_dates=True)

data = load_data('data/prices.csv')

In [13]:
def calculate_return(data: pd.DataFrame) -> pd.DataFrame:
    """Function to return the percentage change per day."""
    return data.pct_change()

returns = calculate_return(data)

    

    

In [14]:
returns.corr()

,EMIM.AS,IUSN.DE,IWDA.AS
EMIM.AS,1.000000,0.732068,0.756767
IUSN.DE,0.732068,1.000000,0.918314
IWDA.AS,0.756767,0.918314,1.000000


In [15]:

returns

,EMIM.AS,IUSN.DE,IWDA.AS
Date,,,
2018-04-26,NaN,NaN,NaN
2018-04-27,0.010412,0.001064,0.004364
2018-04-30,0.005312,0.003899,0.004345
2018-05-02,-0.003894,0.008121,0.001664
2018-05-03,-0.019784,-0.007938,-0.012625
...,...,...,...
2026-06-04,-0.014915,0.005063,-0.000323
2026-06-05,-0.034740,-0.009851,-0.004359
2026-06-08,-0.001826,-0.000791,-0.005514


### Antwoord vraag 1

De correlatiematrix over de dagrendementen (april 2018 t/m juni 2026, 2060 handelsdagen):

|         | EMIM.AS | IUSN.DE | IWDA.AS |
|---------|---------|---------|---------|
| EMIM.AS | 1.00    | 0.73    | 0.76    |
| IUSN.DE | 0.73    | 1.00    | 0.92    |
| IWDA.AS | 0.76    | 0.92    | 1.00    |

**Wat de getallen zeggen.** De correlatie tussen IWDA en IUSN is met 0.92 zeer sterk: 
small caps bewegen op dagbasis vrijwel volledig mee met de wereldindex, ondanks dat er 
geen enkele aandelenoverlap tussen de twee fondsen bestaat. Inhoudelijk verschillende 
fondsen blijken dus niet automatisch gedragsmatig verschillende fondsen. EMIM beweegt 
met 0.73 (t.o.v. IUSN) en 0.76 (t.o.v. IWDA) duidelijk losser: nog steeds sterk 
positief gecorreleerd, maar met merkbaar meer eigen beweging dan small caps.

**Betekenis voor mijn allocatie.** Van mijn twee 20%-posities doet EMIM aantoonbaar 
het meeste diversificatiewerk. De small-cap-positie voegt op dagbasis nauwelijks 
zelfstandige beweging toe; als die positie zijn plek in de portefeuille verdient, 
moet dat uit extra rendement komen en niet uit spreiding. Dat is precies wat vraag 2 
gaat meten.

**Kanttekening bij de maat.** Correlatie meet of fondsen samen bewegen, niet hoe hard. 
IUSN kan voor 92% meebewegen met IWDA en tegelijk volatieler zijn; dat onderscheid 
(samenhang versus beweeglijkheid) komt in vraag 2 aan bod.

**Mijn conclusie:** Het eerste signaal uit het berekenen van de correlatiematrix van deze dataset duidt aan dat ze enigzins tegen verwachting in sterk met elkaar correleren. Voor vraag 2 betekent dit dat ik verwacht dat mijn diversificatie tov een wereldindex de EMIM sterker onafhankelijk beweegt dan de IUSN, en dat dit een iets zwakker effect voor spreiding betekent. Opzich is dit niet super erg want een wereldindex is al een sterke spreiding maar wel een interessante bevinding na het doen van deze analyse. 

## Vraag 2: Mijn mix versus 100% IWDA

**Vraag:** Wat levert mijn 60/20/20-mix (IWDA/EMIM/IUSN) qua rendement en risico op 
vergeleken met simpelweg 100% IWDA over dezelfde periode? Krijg ik betaald voor het 
extra risico, of niet?

**Verwachting vooraf (uit vraag 1):** De spreidingswinst moet vooral van EMIM komen; 
IUSN moet zijn plek eerder in extra rendement bewijzen, want het beweegt vrijwel 
volledig mee met de wereldindex (correlatie 0.92).

**Aanpak:** Van de losse fondsrendementen naar één gewogen portefeuillerendement 
(60% IWDA, 20% EMIM, 20% IUSN). Daarna voor zowel de mix als 100% IWDA twee 
kengetallen berekenen: gemiddeld rendement en standaarddeviatie (risico), beide 
geannualiseerd zodat de getallen iets zeggen op jaarbasis.

**Op te zoeken pandas-operaties:**
- Kolommen wegen en optellen tot één reeks (gewogen som)
- Gemiddelde en standaarddeviatie over een kolom (ken ik conceptueel al van np.mean 
  en np.std, nu de pandas-variant)
- Annualiseren: van dagcijfers naar jaarcijfers (conventie opzoeken)


In [ ]:
def weighed_sum(returns: pd.DataFrame, weights: list[float]) -> pd.Series:
    """Bereken het portefeuillerendement per dag als gewogen som van de fondsrendementen; weights volgt de kolomvolgorde van returns."""
    return weights[0] * returns[returns.columns[0]] + weights[1] * returns[returns.columns[1]] + weights[2] * returns[returns.columns[2]] 
    
weighed_sum(returns, weights=[0.2, 0.2, 0.6])